# 03b · Attention-MIL evaluation & interpretability  [CPU]
Per-fruit metrics with 95% CIs (fruits are independent bags → Wilson + bootstrap), decision threshold tuned on validation, and an attention figure showing which slices drove each decision.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()/'nbpkg'))
import numpy as np, pandas as pd
from config import CFG
import dataset as ds, eval_core as ec, mil
CFG.out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
import pickle
pred=pickle.load(open(CFG.out_dir/'mil_predictions.pkl','rb'))
print('backbones:',list(pred))

## Step 1 — tune decision threshold on VALIDATION, evaluate TEST with CIs

In [ ]:
res={}
for bb,d in pred.items():
    t,_=ec.select_prob_threshold(d['val']['y'], d['val']['probs'], objective=CFG.threshold_objective)
    res[bb]=ec.evaluate_bags(d['test']['y'], d['test']['probs'], threshold=t, n_boot=CFG.n_boot)
tbl=ec.bag_results_to_frame(res); display(tbl)
tbl.to_csv(CFG.out_dir/'mil_table.csv',index=False)
open(CFG.out_dir/'mil_table.md','w').write(tbl.to_markdown(index=False))

## Step 2 — confusion matrices

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,len(res),figsize=(4*len(res),3.4))
for a,(bb,r) in zip(np.atleast_1d(ax),res.items()):
    c=r['confusion']; M=np.array([[c['tn'],c['fp']],[c['fn'],c['tp']]])
    a.imshow(M,cmap='Blues')
    for (i,j),v in np.ndenumerate(M): a.text(j,i,v,ha='center',va='center')
    a.set_xticks([0,1]);a.set_xticklabels(['Ctrl','Inf']);a.set_yticks([0,1])
    a.set_yticklabels(['Ctrl','Inf']);a.set_title(bb)
plt.tight_layout();plt.savefig(CFG.out_dir/'mil_confusion.png',dpi=150);plt.show()

## Step 3 — interpretability: which slices did attention focus on?
For the best backbone (highest test AUC), show the top-attended slices of a few correctly-called infested test fruit. They should land on the visible galleries.

In [ ]:
from PIL import Image
best=max(res, key=lambda b: res[b]['AUC'][0]); print('best backbone:',best)
fruits=ds.build_fruit_index(CFG.data_root, fruit_key=CFG.fruit_key)[0]
by={f.fruit_id:f for f in fruits}
d=pred[best]['test']; ids=d['ids']; y=d['y']; probs=d['probs']; attn=d['attn']
inf_idx=[k for k in range(len(ids)) if y[k]==1 and probs[k]>=0.5]
for k in inf_idx[:3]:
    fid=ids[k]; paths=ds.slice_paths(by[fid].scans[0].directory)
    top=np.argsort(attn[k])[::-1][:6]
    fig,ax=plt.subplots(1,6,figsize=(14,2.4))
    for a,j in zip(ax,top):
        a.imshow(Image.open(paths[j]),cmap='gray'); a.axis('off')
        a.set_title(f'w={attn[k][j]:.2f}',fontsize=8)
    fig.suptitle(f'{fid}  (p_inf={probs[k]:.2f}) — top-attended slices'); plt.show()

Report these MIL metrics as the primary result: no manual curation, no arbitrary voting threshold, fruit-level evaluation with CIs, and attention maps as evidence the model attends to genuine galleries. State the frozen-backbone choice in Methods.